BOXBOX - F1 Strategy Intelligence Dashboard

Phase 2: Data Cleaning & Feature Engineering


In [1]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
warnings.filterwarnings('ignore')

In [2]:
os.chdir(r'C:\Users\adity\Desktop\BoxBox')
os.makedirs('data/processed', exist_ok=True)

logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase2_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)


CIRCUIT TYPE MAPPING

Manually assigned based on known F1 circuit characterstics

Street = temporary street circuits with walls close to track

Highspeed = long straights, fast corners, low downforce

Technical = tight corners, heavy braking, high downforce

Balanced = mix of all characterstics

In [3]:
CIRCUIT_TYPES = {
    'Bahrain': 'Technical',
    'Saudi Arabia': 'HighSpeed',
    'Australia': 'Street',
    'Japan': 'Technical',
    'China': 'Balanced',
    'Miami': 'Balanced',
    'Emilia Romagna': 'Technical',
    'Monaco': 'Street',
    'Canada': 'Balanced',
    'Spain': 'Balanced',
    'Austria': 'Technical',
    'Great Britain': 'HighSpeed',
    'Hungary': 'Technical',
    'Belgium': 'HighSpeed',
    'Netherlands': 'Technical',
    'Italy': 'HighSpeed',
    'Azerbaijan': 'Street',
    'Singapore': 'Street',
    'United States': 'Technical',
    'Mexico': 'HighSpeed',
    'Brazil': 'Technical',
    'Las Vegas': 'Street',
    'Qatar': 'Technical',
    'Abu Dhabi': 'Balanced'
}

Loading Raw Data

Loading all the 5 csv files that we got in phase 1, and loading them here at the top so every function below has access to them without re-reading from disk multiple times

In [4]:
def load_raw_data():
    log.info("Loading files...")

    base = r'C:\Users\adity\Desktop\BoxBox\data\raw'

    laps = pd.read_csv(os.path.join(base, 'raw_laps.csv'))
    pits = pd.read_csv(os.path.join(base, 'pit_stops.csv'))
    results = pd.read_csv(os.path.join(base, 'race_results.csv'))
    weather = pd.read_csv(os.path.join(base, 'weather.csv'))

    log.info(f" raw_laps: {len(laps)} rows")
    log.info(f" pit_stops: {len(pits)} rows")
    log.info(f" race_results: {len(results)} rows")
    log.info(f" weather: {len(weather)} rows")

    return laps, pits, results, weather

CLEAN LAP DATA

Removes all laps that don't represent true race pace

Also extracts safety car lap information before removing those laps, so we can calculate SC probability per circuit

In [5]:
def clean_laps(laps):
    log.info("CLEANING LAP DATA")
    log.info("-" * 50)

    df = laps.copy()
    original_count = len(df)

    # STEP A: Extract safety car laps FIRST, before any
    # filtering removes them. FastF1 marks SC/VSC laps as
    # IsAccurate=False, so extracting after that filter
    # would find zero SC laps every time.
    if 'TrackStatus' in df.columns:
        df['TrackStatus'] = df['TrackStatus'].astype(str)
        sc_laps = df[df['TrackStatus'].isin(['4', '6'])].copy()
        sc_laps_per_circuit = sc_laps.groupby('CircuitName').size()
        total_laps_per_circuit = df.groupby('CircuitName').size()
        sc_probability = (sc_laps_per_circuit / total_laps_per_circuit).fillna(0)
        log.info(f"  Safety car laps extracted: {len(sc_laps)}")
    else:
        sc_probability = pd.Series(dtype=float)

    # STEP B: Drop rows with missing lap time
    df = df.dropna(subset=['LapTimeSeconds'])
    log.info(f"  After dropping missing lap times: {len(df)}")

    # STEP C: Remove laps FastF1 flags as inaccurate
    if 'IsAccurate' in df.columns:
        df = df[df['IsAccurate'] == True]
    log.info(f"  After IsAccurate filter: {len(df)}")

    # STEP D: Filter down to green flag laps only
    if 'TrackStatus' in df.columns:
        df = df[df['TrackStatus'] == '1']
    log.info(f"  After green flag filter: {len(df)}")

    # STEP E: Remove pit in and pit out laps
    if 'IsPitInLap' in df.columns:
        df = df[df['IsPitInLap'] == False]
    if 'IsPitOutLap' in df.columns:
        df = df[df['IsPitOutLap'] == False]
    log.info(f"  After pit lap removal: {len(df)}")

    # STEP F: Keep only dry weather compounds
    # This is what correctly excludes Brazil (wet-only race)
    valid_compounds = ['SOFT', 'MEDIUM', 'HARD']
    df = df[df['Compound'].isin(valid_compounds)]
    log.info(f"  After compound filter: {len(df)}")

    # STEP G: Remove rows with missing TyreLife
    df = df.dropna(subset=['TyreLife'])
    df = df[df['TyreLife'] > 0]
    log.info(f"  After TyreLife filter: {len(df)}")

    # STEP H: Remove statistical outliers per circuit
    clean_groups = []
    for circuit, group in df.groupby('CircuitName'):
        mean = group['LapTimeSeconds'].mean()
        std = group['LapTimeSeconds'].std()
        upper = mean + 2 * std
        filtered = group[group['LapTimeSeconds'] <= upper]
        clean_groups.append(filtered)

    df = pd.concat(clean_groups, ignore_index=True)
    log.info(f"  After outlier removal: {len(df)}")
    log.info(f"  Total removed: {original_count - len(df)} laps")

    return df, sc_probability

CLEAN PIT STOP DATA 

Removes pit stops with unrealistic durations and calculates average pit loss time per circuit for the optimizer

In [6]:
def clean_pit_stops(pits):
    log.info("CLEANING PIT STOP DATA")
    log.info("-" * 50)

    df = pits.copy()
    original_count = len(df)

    # Remove rows where pit duration is missing
    df = df.dropna(subset=['PitDurationSeconds'])

    '''Remove unrealistic dusraations
    Under 1.5s = data error (stationary change takes
    ~2.5s minimum) Over 60s = drive-through penalty or
    very unusual event, not a normal strategy stop'''
    df = df[
        (df['PitDurationSeconds'] >= 1.5) &
        (df['PitDurationSeconds'] <= 60.0)
    ]

    log.info(f" Removed {original_count - len(df)} invalid pit stops")
    log.info(f" Clean pit stops remaining: {len(df)}")

    '''Calculate average pit loss time per circuit
    Pit loss = stationary time + pit lane traversal time
    We estimate total pit loss as stationary time + 17 sec
    (average pit traversal based on typical F1 pit lane lengths)
    This gives us a per-circuit pit loss estimate better than
    the fixed 22-second value we used in Phase 3 before'''
    df['EstimatedPitLoss'] = df['PitDurationSeconds'] + 17.0

    pit_loss_per_circuit = df.groupby('CircuitName').agg(
        AvgPitLoss = ('EstimatedPitLoss', 'mean'),
        AvgStationaryTime = ('PitDurationSeconds', 'mean'),
        TotalStops = ('PitDurationSeconds', 'count')
    ).round(2)

    log.info("\n Pit loss per circuit:")
    log.info(pit_loss_per_circuit.to_string())

    return df, pit_loss_per_circuit

MERGE WEATHER ONTO LAPS

Weather is sampled every few minutes, not every lap, so we will merge it onto lap data by matching the closest timestamp. 

Each lap gets the weather reading that was closest to the time when that lap occurred

In [7]:
def merge_weather(laps_clean, weather):
    log.info("MERGING WEATHER DATA")
    log.info("-" * 50)

    weather_cols = ['AirTemp', 'TrackTemp', 'Humidity',
                    'Windspeed', 'Rainfall', 'CircuitName', 'TimeSeconds']
    
    available_cols = [c for c in weather_cols if c in weather.columns]
    weather_subset = weather[available_cols].copy()

    merged_groups = []

    for circuit in laps_clean['CircuitName'].unique():
        lap_group = laps_clean[laps_clean['CircuitName'] == circuit].copy()
        wx_group = weather_subset[
            weather_subset['CircuitName'] == circuit
        ].copy()

        if wx_group.empty or 'TimeSeconds' not in wx_group.columns:
            # If no weather data for this circuit, fill with NaN
            for col in ['AirTemp', 'TrackTemp', 'Humidity',
                        'WindSpeed', 'Rainfall']:
                if col not in lap_group.columns:
                    lap_group[col] = np.nan
            merged_groups.append(lap_group)
            continue

        '''For each lap we need a timestamp to match against
        weather. Thats where we use lap number as a proxy, 
        multiply by average lap time to get approximate session 
        time in seconds
        This is an approximation but good enough for weather 
        matching since weather changes slowly over a race'''
        if 'LapTimeSeconds' in lap_group.columns:
            lap_group['ApproxSessionTime'] = (
                lap_group['LapNumber'] * lap_group['LapTimeSeconds']
            )
        else:
            lap_group['ApproxSessionTime'] = lap_group['LapNumber'] * 90
        
        # Merge using nearest timestamp match
        # pd.merge_asof matches each lap to the closest weather reading 
        lap_sorted = lap_group.sort_values('ApproxSessionTime')
        wx_sorted = wx_group.sort_values('TimeSeconds')

        wx_cols_to_merge = [c for c in ['AirTemp', 'TrackTemp', 'Humidity',
                                         'WindSpeed', 'Rainfall', 'TimeSeconds']
                            if c in wx_sorted.columns]
        merged = pd.merge_asof(
            lap_sorted,
            wx_sorted[wx_cols_to_merge],
            left_on='ApproxSessionTime',
            right_on='TimeSeconds',
            direction='nearest'
        )

        merged_groups.append(merged)

    result = pd.concat(merged_groups, ignore_index=True)

    log.info(f" Weather merged onto {len(result)} laps")
    log.info(f" Circuits with weather: "
            f"{result['TrackTemp'].notna().sum()} laps have track temp")
    
    return result

ENGINEER FEATURES 

Creates all new calculated columns on the cleaned, 
weather-merged lap dataset

In [8]:
def engineer_features(df, sc_probability, pit_loss_per_circuit):
    log.info("ENGINEER FEATURES")
    log.info("-" * 50)

    result = df.copy()

    '''FEATURE 1: Feul corrected lap time
    Cars start with ~100kg  feul, burn ~1.8kg per lap
    Weight reduction = 1.8kg * lap_number (approximate)
    Performance gain = 0.03 seconds per kg lighter
    So feul effect = 0.03 * 1.8 * lap_number = 0.054s
    per lap
    We ADD this back to lap time to isolate tire degradation
    from the feul burn effect'''
    result['FuelCorrectedLapTime'] = (
        result['LapTimeSeconds'] + 0.054 * result['LapNumber']
    )
    log.info(" Fuel Corrected Lap Time")

    '''Feature 2: Stint Progress (0 to 1)
    How far through the current stint is the driver
    We calculate max TyreLife per driver per stint per circuit
    the divide current TyreLife by that maximum'''

    stint_max = result.groupby(
        ['CircuitName', 'Driver', 'Stint']
    )['TyreLife'].transform('max')

    result['StintProgress'] = result['TyreLife'] / stint_max
    result['StintProgress'] = result['StintProgress'].clip(0, 1)
    log.info(" Stint Progress")

    '''Feature 3: COMPOUND AGE RATIO
    TyreLife divided by the maximum observed life for that
    compound at that circuit across all drivers'''
    compound_max_life = result.groupby(
        ['CircuitName', 'Compound']
    )['TyreLife'].transform('max')

    result['CompoundAgeRatio'] = result['TyreLife'] / compound_max_life
    result['CompoundAgeRatio'] = result['CompoundAgeRatio'].clip(0, 1)
    log.info(" Compound Age Ratio")

    '''FEATURE 4: NORMALIZED LAP TIME
    Lap Time relative to the fastest lap at that circuit
    A value of 1.0 = fastest lap, 1.02 = 2% slower than fastest
    This makes the circuits comparable in ML models'''
    circuit_min_time = result.groupby(
        'CircuitName'
    )['LapTimeSeconds'].transform('min')

    result['NormalizedLapTime'] = result['LapTimeSeconds'] / circuit_min_time
    log.info(" Normalized Lap Time")

    '''FEATURE 5: DEGRADATION ACCELERATION FLAG
    Is the tire degrading faster in the second half of
    the stint than the first half
    We calculate this per driver per stint per circuit'''
    def calc_deg_acceleration(group):
        if len(group) < 6:
            group['DegAcceleration'] = False
            return group
        
        midpoint = group['TyreLife'].median()
        first_half = group[group['TyreLife'] <= midpoint]
        second_half = group[group['TyreLife'] > midpoint]

        if len(first_half) < 2 or len(second_half) < 2:
            group['DegAcceleration'] = False
            return group
        
        '''Calculate slope (degradation rate) for each half
        using simple rise/run approximation'''
        first_slope = (
            first_half['LapTimeSeconds'].iloc[-1] -
            first_half['LapTimeSeconds'].iloc[0]
        ) / max(len(first_half), 1)

        second_slope =(
            second_half['LapTimeSeconds'].iloc[-1] -
            second_half['LapTimeSeconds'].iloc[0]
        ) / max(len(second_half), 1) 

        #Flag True if seconds half degrades more than 20% faster
        group['DegAcceleration'] = second_slope > first_slope * 1.2
        return group
    result = result.groupby(
        ['CircuitName', 'Driver', 'Stint'],
        group_keys = False
    ).apply(calc_deg_acceleration)
    log.info(" Degradation Acceleration")

    '''FEATURE 6: CIRCUIT TYPE
    Categorial label assigned from our manual mapping above'''
    result['CircuitType'] = result['CircuitName'].map(CIRCUIT_TYPES)
    result['CircuitType'] = result['CircuitType'].fillna('Balanced')
    log.info(" Circuit Type")

    '''FEATURE 7: SAFETY CAR PROBABILITY
    What fractions of laps at this circuit were under SC/VSC'''
    result['SafetyCarProbability'] = result['CircuitName'].map(sc_probability)
    result['SafetyCarProbability'] = result['SafetyCarProbability'].fillna(0)
    log.info(" Safety Car Probability")

    '''FEATURE 8: PIT LOSS TIME PER CIRRCUIT
    Average total time lost per pit stop at each circuit'''
    if not pit_loss_per_circuit.empty:
        pit_loss_map = pit_loss_per_circuit['AvgPitLoss'].to_dict()
        result['CircuitPitLoss'] = result['CircuitName'].map(pit_loss_map)
        result['CircuitPitLoss'] = result['CircuitPitLoss'].fillna(22.0)
    else:
        result['CircuitPitLoss'] = 22.0
    log.info(" Circuit Pit Loss")

    '''FEATURE 9: COMPOUND ENCODED
    ML Models need numbers not strings - encode compound as integer
    SOFT = 2(fastest) | MEDIUM = 1 | HARD = 0(slowest)'''
    compound_map = {'SOFT': 2, 'MEDIUM': 1, 'HARD': 0}
    result['CompoundEncoded'] = result['Compound'].map(compound_map)
    log.info(" Compound Encoded")

    '''FEATURE 10: CIRCUIT TYPE ENCODED
    Same as we did with the tyre compounds
    Street = 0 | Technical = 1 | Balanced = 2 | HighSpeed = 3'''
    circuit_type_map = {
        'Street': 0, 'Technical': 1,
        'Balanced': 2, 'HighSpeed': 3
    }
    result['CircuitTypeEncoded'] = result['CircuitType'].map(circuit_type_map)
    log.info(" Circuit Type Encoded")

    log.info(f"\n Total features in dataset: {len(result.columns)}")
    log.info(f" Total clean laps: {len(result)}")

    return result

BUILD FEATURE STORE

Creates a circuit level summary table - one row per circuit

This is what next phases will use as a circuit context rather than going back to the full lap-level dataset

In [9]:
def build_feature_store(engineered_laps, pit_loss_per_circuit, sc_probability):
    log.info(" BUILDING FEATURE STORE")
    log.info("-" * 50)

    circuits = engineered_laps['CircuitName'].unique()
    rows =[]

    for circuit in sorted(circuits):
        circuit_laps = engineered_laps[
            engineered_laps['CircuitName'] == circuit
        ]

        row = {
            'CircuitName': circuit,
            'CircuitType': CIRCUIT_TYPES.get(circuit, 'Balanced'),
            'CircuitTypeEncoded' : {
                'Street': 0, 'Technical': 1,
                'Balanced': 2, 'HighSpeed': 3
                }.get(CIRCUIT_TYPES.get(circuit, 'Balanced'), 2),
            
            # Average conditions
            'AvgTrackTemp' : circuit_laps['TrackTemp'].mean()
                if 'TrackTemp' in circuit_laps.columns else np.nan,
            'AvgAirTemp' : circuit_laps['AirTemp'].mean()
                if 'AirTemp' in circuit_laps.columns else np.nan,
            'AvgHumidity' : circuit_laps['Humidity'].mean()
                if 'Humidity' in circuit_laps.columns else np.nan,
            
            #Race Characterstics
            'TotalRaceLaps' : int(circuit_laps['LapNumber'].max()),
            'SafetyCarProbability': sc_probability.get(circuit, 0),
            'AvgPitLoss' : pit_loss_per_circuit.loc[circuit, 'AvgPitLoss']
                if circuit in pit_loss_per_circuit.index else 22.0,

            #Compound availability in this circuit
            'HasSoft': int('SOFT' in circuit_laps['Compound'].values),
            'HasMedium' : int('MEDIUM' in circuit_laps['Compound'].values),
            'HasHard' : int('HARD' in circuit_laps['Compound'].values),

            #Degradation indicators (rough)
            'AvgStintLength' : circuit_laps.groupby(
                ['Driver', 'Stint']
            )['TyreLife'].max().mean(),
            'MaxObservedTyreLife' : circuit_laps['TyreLife'].max(),

            #Pace reference
            'FastestLapSeconds' : circuit_laps['LapTimeSeconds'].min(),
            'AvgRaceLapSeconds' : circuit_laps['LapTimeSeconds'].mean(),
        }

        rows.append(row)
    
    feature_store = pd.DataFrame(rows)

    log.info(f" Feature store built: {len(feature_store)} circuits")
    log.info(feature_store[
        ['CircuitName', 'CircuitType', 'AvgPitLoss',
        'SafetyCarProbability', 'TotalRaceLaps']
    ].to_string(index=False))

    return feature_store

MAIN PIPELINE

In [10]:
def main():
    log.info(" BOXBOX Phase 2 : Data Cleaning & Feature Engineering")
    log.info('-' * 50)

    base_out = r'C:\Users\adity\Desktop\BoxBox\data\processed'

    #Load
    laps, pits, results, weather = load_raw_data()

    #Clean
    laps_clean, sc_probability = clean_laps(laps)
    pits_clean, pit_loss_per_circuit = clean_pit_stops(pits)

    #Merge weather
    laps_with_weather = merge_weather(laps_clean, weather)

    #Engineer features
    engineered = engineer_features(
        laps_with_weather, sc_probability, pit_loss_per_circuit
    )

    print(sc_probability)
    print(sc_probability.get('Canada', 0))
    #Build feature store
    feature_store = build_feature_store(
        engineered, pit_loss_per_circuit, sc_probability
    )

    #Save outputs
    engineered_path = os.path.join(base_out, 'engineered_laps.csv')
    feature_store_path = os.path.join(base_out, 'feature_store.csv')
    pits_clean_path = os.path.join(base_out, 'pit_stops_clean.csv')

    engineered.to_csv(engineered_path, index = False)
    feature_store.to_csv(feature_store_path, index = False)
    pits_clean.to_csv(pits_clean_path, index = False)

    log.info("-" * 50)
    log.info("Phase 2 complete")

    for name, path in [
        ('Engineered Laps', engineered_path),
        ('Feature Store', feature_store_path),
        ('Clean Pit Stops', pits_clean_path),
    ]:
        if os.path.exists(path):
            size_mb = os.path.getsize(path) / (1024 * 1024)
            df_check = pd.read_csv(path)
            log.info(f" {name}: {len(df_check)} rows | "
                    f"{len(df_check.columns)} cols | {size_mb:.1f} MB")
        
        else:
            log.warning(f" {name}: NOT SAVED")
    
    log.info(f" Total features engineered: 10")
    log.info(f" Circuits in dataset: "
            f"{engineered['CircuitName'].nunique()}")
    log.info(f" Clean laps: {len(engineered)}")


if __name__ == '__main__':
    main()

2026-07-28 07:16:13,622 - INFO -  BOXBOX Phase 2 : Data Cleaning & Feature Engineering
2026-07-28 07:16:13,623 - INFO - --------------------------------------------------
2026-07-28 07:16:13,624 - INFO - Loading files...
2026-07-28 07:16:13,735 - INFO -  raw_laps: 26604 rows
2026-07-28 07:16:13,736 - INFO -  pit_stops: 849 rows
2026-07-28 07:16:13,736 - INFO -  race_results: 479 rows
2026-07-28 07:16:13,737 - INFO -  weather: 3690 rows
2026-07-28 07:16:13,737 - INFO - CLEANING LAP DATA
2026-07-28 07:16:13,738 - INFO - --------------------------------------------------
2026-07-28 07:16:13,749 - INFO -   Safety car laps extracted: 569
2026-07-28 07:16:13,756 - INFO -   After dropping missing lap times: 26381
2026-07-28 07:16:13,762 - INFO -   After IsAccurate filter: 23557
2026-07-28 07:16:13,771 - INFO -   After green flag filter: 22999
2026-07-28 07:16:13,779 - INFO -   After pit lap removal: 22999
2026-07-28 07:16:13,786 - INFO -   After compound filter: 21354
2026-07-28 07:16:13,791 

CircuitName
Abu Dhabi         0.000000
Australia         0.010020
Austria           0.000000
Azerbaijan        0.015416
Bahrain           0.000000
Belgium           0.000000
Brazil            0.042328
Canada            0.090409
China             0.106589
Emilia Romagna    0.000000
Great Britain     0.000000
Hungary           0.000000
Italy             0.000000
Japan             0.000000
Las Vegas         0.000000
Mexico            0.059259
Miami             0.051305
Monaco            0.000000
Netherlands       0.000000
Qatar             0.111347
Saudi Arabia      0.019978
Singapore         0.000000
Spain             0.000000
United States     0.017941
dtype: float64
0.09040880503144653


2026-07-28 07:16:15,540 - INFO - --------------------------------------------------
2026-07-28 07:16:15,541 - INFO - Phase 2 complete
2026-07-28 07:16:15,642 - INFO -  Engineered Laps: 20887 rows | 46 cols | 5.8 MB
2026-07-28 07:16:15,648 - INFO -  Feature Store: 23 rows | 16 cols | 0.0 MB
2026-07-28 07:16:15,677 - INFO -  Clean Pit Stops: 769 rows | 11 cols | 0.1 MB
2026-07-28 07:16:15,677 - INFO -  Total features engineered: 10
2026-07-28 07:16:15,680 - INFO -  Circuits in dataset: 23
2026-07-28 07:16:15,682 - INFO -  Clean laps: 20887
